# Brain-Act Cached Analysis Only (Parallel)

This notebook **does not re-run simulations**.
It loads cached `*.npz` outputs from `notebooks/outputs/ba_dual2min/sims` and computes:

- LZc (rates + BOLD)
- PCI Casali-like (rates + BOLD)
- Brain states / occupancy / SCFC coupling (rates + BOLD), using legacy pooled states **within each scenario only**

Progress is printed for every completed job and appended to `run_progress.log`.


In [ ]:
from __future__ import annotations

import os
import sys
import subprocess
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Match simulation notebook runtime guards
os.environ.setdefault('MPLCONFIGDIR', str(Path('/tmp/matplotlib-cache').resolve()))
os.environ.setdefault('XDG_CACHE_HOME', str(Path('/tmp/xdg-cache').resolve()))
os.environ.setdefault('TVB_USER_HOME', str((PROJECT_ROOT / '.tvb-temp').resolve()))
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('VECLIB_MAXIMUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from tvbtoolkit.workflows.brain_act_cached_analysis import (
    run_cached_dual_domain_analysis,
    finalize_cached_analysis_tables,
)


In [ ]:
# Configuration
RUN_NAME = 'ba_dual2min'
OUT_ROOT = PROJECT_ROOT / 'notebooks' / 'outputs' / RUN_NAME
SIM_DIR = OUT_ROOT / 'sims'
METRICS_DIR = OUT_ROOT / 'res'
FIG_DIR = OUT_ROOT / 'figs'
RUN_LOG_PATH = OUT_ROOT / 'run_progress.log'
DATASET_ROOT = PROJECT_ROOT / 'data' / 'doc_patients_new_data' / 'converted_structural'
PLOT_SCRIPT = PROJECT_ROOT / 'scripts' / 'plot_stats_from_saved_rates.py'

N_STATES = 5
PCI_WINDOW_RATE_MS = 300.0
PCI_WINDOW_BOLD_MS = 30000.0
STATE_MODE = 'legacy_pooled'  # pooled within each scenario (independent experiments)
POOLED_RATE_MAX_ROWS_PER_JOB = 120
POOLED_RANDOM_SEED = 0

USE_PROCESSES = True
N_JOBS = max(1, int(round((os.cpu_count() or 8) * 0.85)))
PROGRESS_EVERY = 1  # print every completed job

for d in (METRICS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print({
    'out_root': str(OUT_ROOT),
    'sim_dir': str(SIM_DIR),
    'metrics_dir': str(METRICS_DIR),
    'fig_dir': str(FIG_DIR),
    'run_log': str(RUN_LOG_PATH),
    'dataset_root': str(DATASET_ROOT),
    'n_jobs': N_JOBS,
    'use_processes': USE_PROCESSES,
    'state_mode': STATE_MODE,
})

if not SIM_DIR.exists():
    raise FileNotFoundError(f'Cached sim directory not found: {SIM_DIR}')


In [ ]:
# Run cached analysis (parallel subject/scenario jobs)
metrics_df, states_df, errors_df = run_cached_dual_domain_analysis(
    sim_dir=SIM_DIR,
    dataset_root=DATASET_ROOT,
    n_states=N_STATES,
    pci_window_rate_ms=PCI_WINDOW_RATE_MS,
    pci_window_bold_ms=PCI_WINDOW_BOLD_MS,
    n_jobs=N_JOBS,
    use_processes=USE_PROCESSES,
    show_progress=True,
    progress_every=PROGRESS_EVERY,
    log_path=RUN_LOG_PATH,
    state_mode=STATE_MODE,
    pooled_rate_max_rows_per_job=POOLED_RATE_MAX_ROWS_PER_JOB,
    pooled_random_seed=POOLED_RANDOM_SEED,
)

print({'metrics_rows_raw': int(metrics_df.shape[0]), 'state_rows_raw': int(states_df.shape[0]), 'errors': int(errors_df.shape[0])})
if not errors_df.empty:
    display(errors_df.head(20))


In [ ]:
# Attach source metadata + coma subgroup labels, then save canonical CSVs
metrics_df, states_df, subject_groups_df = finalize_cached_analysis_tables(
    metrics_df=metrics_df,
    states_df=states_df,
    dataset_root=DATASET_ROOT,
)

metrics_csv = METRICS_DIR / 'dual_domain_metrics.csv'
states_csv = METRICS_DIR / 'dual_domain_state_rows.csv'
subject_groups_csv = METRICS_DIR / 'subject_groups.csv'
errors_csv = METRICS_DIR / 'analysis_cached_errors.csv'

metrics_df.to_csv(metrics_csv, index=False)
states_df.to_csv(states_csv, index=False)
subject_groups_df.to_csv(subject_groups_csv, index=False)
if not errors_df.empty:
    errors_df.to_csv(errors_csv, index=False)

print({
    'metrics_csv': str(metrics_csv),
    'states_csv': str(states_csv),
    'subject_groups_csv': str(subject_groups_csv),
    'errors_csv': str(errors_csv) if errors_df.shape[0] else '(none)',
    'metrics_rows': int(metrics_df.shape[0]),
    'state_rows': int(states_df.shape[0]),
})


In [ ]:
# Plot + stats pass from saved CSVs (Figure 1/2/3 + stats tables)
cmd = [sys.executable, str(PLOT_SCRIPT), '--root', str(OUT_ROOT)]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

print('Generated figures:')
for p in sorted(FIG_DIR.glob('fig*.svg')):
    print(' -', p)
for p in sorted(FIG_DIR.glob('fig*.pdf')):
    print(' -', p)


In [ ]:
# Quick preview
display(metrics_df.head())
display(states_df.head())

stats_files = sorted(METRICS_DIR.glob('stats_*with_stars.csv'))
print('Stats files:')
for p in stats_files:
    print(' -', p)


In [ ]:
# Tail run log (latest lines)
if RUN_LOG_PATH.exists():
    lines = RUN_LOG_PATH.read_text().splitlines()
    print('
'.join(lines[-80:]))
else:
    print('No run log found at', RUN_LOG_PATH)
